### **Processing prev_data**

In [23]:
import json
from datetime import datetime

def fetch_json_data(file_path, output_file_path="cust_data.json"):
    """
    Fetches recipe data from a local JSON file, processes it, and returns only the relevant fields.
    """
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)

        all_recipes = []

        # Iterate over each entry in the JSON file
        for entry in data:
            recipe_info_list = entry.get("recipe_info", [])  # Ensure it's a list

            # Ensure recipe_info_list is actually a list before proceeding
            if not isinstance(recipe_info_list, list):
                print(f"Skipping entry due to unexpected format: {entry}")
                continue

            # Extract and format delivery date
            delivery_date_raw = entry.get("_id", {}).get("delivery_date", {}).get("$date", {}).get("$numberLong")
            if delivery_date_raw:
                # Convert epoch milliseconds to ISO 8601 format
                delivery_date = datetime.utcfromtimestamp(int(delivery_date_raw) / 1000).isoformat() + "Z"
            else:
                delivery_date = None  # If no valid date, keep it as None

            # Iterate through each recipe_info dictionary in the list
            for recipe_info in recipe_info_list:
                # Extract ingredients and merge with variant ingredients
                ingredients = recipe_info.get("ingredients", [])
                variant_ingredients = []

                # Process the variants if available
                if isinstance(recipe_info.get("variants"), dict):  # Ensure "variants" is a dictionary
                    variant_ingredients = recipe_info["variants"].get("variant_ingredients", [])

                # Merge ingredients from both the recipe_info and variants
                all_ingredients = list(set(ingredients + variant_ingredients))

                # Prepare the recipe data with only the necessary fields
                recipe = {
                    "dish_name": recipe_info.get("dish_name"),
                    "meal_category": recipe_info.get("meal_category"),
                    "description": recipe_info.get("description"),
                    "cuisine": recipe_info.get("cuisine"),
                    "ingredients": all_ingredients,  # Merged ingredients
                    "allergens_contain": recipe_info.get("allergens_contain", []),
                    "meal_type": entry.get("meal_type"),
                    "spice_level": recipe_info.get("spice_level", ""),
                    "is_auto_select": recipe_info.get("is_auto_select"),
                    "rating": recipe_info.get("rating") if "rating" in recipe_info else None,
                    "delivery_date": delivery_date  # Add formatted date
                }

                # Add the processed recipe data to the list
                all_recipes.append(recipe)

        # If an output file path is provided, save the processed data to that file
        if output_file_path:
            with open(output_file_path, 'w') as output_file:
                json.dump(all_recipes, output_file, indent=4)
            print(f"Processed data saved to {output_file_path}")

        return all_recipes

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error: {str(e)}")
        return []


In [24]:
import os

# Define the input and output directories
input_directory = "user_data/prev_data"
output_directory = "user_data/processed_files"

# Ensure output directory exists
os.makedirs(output_directory, exist_ok=True)

# Get all JSON files in the input directory
input_file_paths = [os.path.join(input_directory, file) for file in os.listdir(input_directory) if file.endswith(".json")]

# Loop through each input file and process it
for input_file_path in input_file_paths:
    file_name = os.path.basename(input_file_path)
    output_file_path = os.path.join(output_directory, f"processed_{file_name}")
    
    processed_data = fetch_json_data(input_file_path, output_file_path)
    
    if processed_data:
        print(f"Processed data for {file_name} saved to {output_file_path}")
    else:
        print(f"Failed to process data for {file_name}.")


Processed data saved to user_data/processed_files/processed_65a677b8062f61836c3dbffc.json
Processed data for 65a677b8062f61836c3dbffc.json saved to user_data/processed_files/processed_65a677b8062f61836c3dbffc.json
Processed data saved to user_data/processed_files/processed_66224ce5566a4aef142a7d41.json
Processed data for 66224ce5566a4aef142a7d41.json saved to user_data/processed_files/processed_66224ce5566a4aef142a7d41.json
Processed data saved to user_data/processed_files/processed_66a934422279daa2850029ae.json
Processed data for 66a934422279daa2850029ae.json saved to user_data/processed_files/processed_66a934422279daa2850029ae.json
Processed data saved to user_data/processed_files/processed_64e77a0e5d497c8a38d8840a.json
Processed data for 64e77a0e5d497c8a38d8840a.json saved to user_data/processed_files/processed_64e77a0e5d497c8a38d8840a.json
Processed data saved to user_data/processed_files/processed_66a799812279daa285f1d82e.json
Processed data for 66a799812279daa285f1d82e.json saved

### Cust Analysis and query generation using prev_data

In [61]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from IPython.display import display, Markdown

def process_and_analyze_json(output_file_path="processed_data.json"):
    """
    Process and analyze the recipe data from a JSON file.
    It will:
    - Extract and merge the ingredients.
    - Analyze top cuisines, spice levels, and user selections.
    - Display visualizations like pie charts.
    - List all user-rated recipes sorted from highest to lowest rating.
    """
    try:
        # Load the processed JSON data directly
        with open(output_file_path, "r") as file:
            recipes = json.load(file)

        # Convert the processed data into a DataFrame
        df = pd.DataFrame(recipes)

        # Check if 'cuisine' column exists
        if 'cuisine' not in df.columns:
            raise KeyError("Missing 'cuisine' column in the dataset")

        # Identify the top cuisine preference
        top_cuisine = df['cuisine'].mode()
        # print(f"Top Cuisine: {top_cuisine}")
        
        # Identify the user's most preferred spice level
        top_spice_level = df['spice_level'].mode()[0]
        # print(f"Top Spice Level: {top_spice_level}")

        # No longer filtering by `is_auto_select`
        user_selected_meals = df.copy()  

        # Display a summary of all meals
        user_selected_meals_summary = user_selected_meals.describe(include='object')
        # print("Summary of All Selected Meals:")
        # display(user_selected_meals_summary)

        # Get the most frequent dish names
        top_dish_names = user_selected_meals['dish_name'].value_counts().reset_index()
        top_n = 5
        # display(Markdown("### Most Frequently Selected Dishes:"))
        # display(top_dish_names.head(top_n))

        # Get the count of each cuisine selected by the user
        cuisine_counts = user_selected_meals['cuisine'].value_counts().reset_index()
        cuisine_counts.columns = ["Cuisine", "Count"]
        # display(Markdown("### Total Cuisine Counts Across all Weeks :"))
        # display(cuisine_counts)

        # Display all user-rated recipes sorted from highest to lowest rating
        if 'rating' in df.columns:
            top_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=False)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(top_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        if 'rating' in df.columns:
            least_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=True)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(least_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        # Pie Chart for Cuisine Preferences
        # sns.set(style="darkgrid")
        # plt.style.use('dark_background')

        # plt.figure(figsize=(4, 4))  
        # plt.pie(
        #     cuisine_counts['Count'], 
        #     labels=cuisine_counts['Cuisine'], 
        #     autopct='%1.1f%%',  
        #     colors=sns.color_palette("tab20", len(cuisine_counts)), 
        #     startangle=90, 
        #     wedgeprops={'edgecolor': 'none'},  
        #     labeldistance=1.1,  
        #     pctdistance=0.85  
        # )
        # plt.title('Cuisines Selected by User', fontsize=8, color='white')  
        # plt.ylabel('')
        # plt.gcf().patch.set_alpha(0)
        # # display(Markdown("### Cuisine Preference Across all Weeks :"))
        # plt.show()

        # Weekly Cuisine Preferences Analysis
        if 'delivery_date' in df.columns:
            df['delivery_date'] = df['delivery_date'].apply(lambda x: str(x) if isinstance(x, dict) else x)
            df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')
        
        min_date = df['delivery_date'].min()
        df['week_number'] = df['delivery_date'].apply(lambda x: (x - min_date).days // 7 + 1)

        cuisine_weekly_counts = df.groupby(['week_number', 'cuisine']).size().reset_index(name='count')
        pivot_table = cuisine_weekly_counts.pivot(index="week_number", columns="cuisine", values="count").fillna(0)
        cuisine_order = cuisine_weekly_counts.groupby("cuisine")["count"].sum().sort_values()
        pivot_table = pivot_table[cuisine_order.index]

        # display(Markdown("### Cuisine Preferences over Weeks :"))
        # display(pivot_table)

        # Generate structured query based on analysis
        top_cuisines = ', '.join(df['cuisine'].mode())  # Get top cuisine(s)
        top_spice_level = df['spice_level'].mode()[0]  # Get top spice level
        # Ensure the first column contains dish names
        top_dishes = ', '.join(top_dish_names.iloc[:6, 0])  # ✅ Corrected way

        # Find highest-rated dishes (handling multiple)
        if 'rating' in df.columns and not df['rating'].isna().all():
            max_rating = df['rating'].max()  # Get the highest rating
            top_rated_dishes = df[df['rating'] == max_rating]['dish_name'].unique()  # Get unique top-rated dishes
            top_rated_dishes = ', '.join(top_rated_dishes[:5])  # Limit to top 5 for readability
        else:
            top_rated_dishes = None
        # Find dishes rated less than 3 (User Dislikes)
        user_dislikes = None
        if 'rating' in df.columns and not df['rating'].isna().all():
            disliked_dishes = df[df['rating'] < 3]['dish_name'].unique()  # Get unique low-rated dishes
            user_dislikes = ', '.join(disliked_dishes[:5])  # Limit to 5 for readability

        # Build the query string
        query = f"Spice Level: {top_spice_level}, Cuisine: {top_cuisines}. Popular Dishes: {top_dishes}"

        if top_rated_dishes:
            query += f". Highest Rated Dishes: {top_rated_dishes}"

        # print("Generated Query:")
        # print(query)

        return df, query, user_dislikes  # Return DataFrame and Query


    except FileNotFoundError:
        print(f"Error: The file '{output_file_path}' was not found.")
        return []
    except KeyError as e:
        print(f"Error: {str(e)}")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{output_file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error: {str(e)}")
        return []

In [62]:
import os
import glob

# Folder containing the JSON files
folder_path = "user_data/processed_files/"

# Get all JSON files in the folder
json_files = glob.glob(os.path.join(folder_path, "*.json"))

# Process each file
for file_path in json_files:
    print(f"\nProcessing: {file_path}")
    df, query, user_dislikes = process_and_analyze_json(file_path)  # Call function for each file
    print(f"Query for {file_path}:\n{query}\n")



Processing: user_data/processed_files/processed_65e1d786f967c10fcba692fd.json
Query for user_data/processed_files/processed_65e1d786f967c10fcba692fd.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Herb Mashed Potato, Cowboy Salad , Chicken a la King & Onion Bread Roll, Beetroot & Quinoa Salad, Coconut Curry and Quinoa, Blackened Protein with Braised Lentils


Processing: user_data/processed_files/processed_6597f5f893f2206fb148f076.json
Query for user_data/processed_files/processed_6597f5f893f2206fb148f076.json:
Spice Level: Medium, Cuisine: American, Mediterranean. Popular Dishes: Korean Style Wrap, Cajun Roasted Chicken Sandwich, Lime Chili Quesadillas , Satay & Jasmin Rice, Lemon Quinoa with Roasted Protein, Maftoul with Zucchini & Minced Protein . Highest Rated Dishes: Coconut Curry and Quinoa, Korean Style Wrap, Shawarma Bowl , Satay & Jasmin Rice, English Loaf


Processing: user_data/processed_files/processed_679bd597ed9daf35d7885142.json
Query for user_data/pr

### ***CSV***

In [65]:
import os
import json
import pandas as pd
import glob

def extract_user_preferences(csv_path="user_data/Customer_Auto_Seletion_Data_50.csv", json_folder="user_data/processed_files/"):
    """
    Extracts user preferences from CSV and JSON files and returns a structured dictionary.
    """
    # Load CSV
    df_csv = pd.read_csv(csv_path)

    # Dictionary to store user preferences
    user_preferences = {}

    # Get all JSON files in the folder
    json_files = glob.glob(os.path.join(json_folder, "*.json"))

    # Process each matching file
    for file_path in json_files:
        # Extract user_id from file name
        user_id = os.path.basename(file_path).replace("processed_", "").replace(".json", "")

        # Find matching row in CSV
        matching_row = df_csv[df_csv.iloc[:, 0] == user_id]

        if not matching_row.empty:
            print(f"\nProcessing: {file_path}")

            # Extract values from CSV row
            row = matching_row.iloc[0]

            # Avoid ingredients (default to empty set if missing)
            user_allergens = set(str(row["avoid_ingredient"]).split(',')) if pd.notna(row["avoid_ingredient"]) else {}

            # Protein category (default to empty string if missing)
            protein_category = row["diet_type"] if pd.notna(row["diet_type"]) else ""

            # Size (default to empty string if missing)
            size = row["variant_size"] if pd.notna(row["variant_size"]) else ""

            # Meal type as a set (default to empty set if missing)
            meal_types = set(str(row["plan"]).split(',')) if pd.notna(row["plan"]) else {}

            # Process JSON file
            df, query, user_dislikes = process_and_analyze_json(file_path)

            # Store results in a dictionary
            user_preferences[user_id] = {
                "user_pref": query.split("Popular Dishes: ")[1] if "Popular Dishes: " in query else "",
                "user_likes": f"{query.split('Cuisine: ')[1].split('.')[0]} cuisine" if "Cuisine: " in query else "",
                "user_allergens": user_allergens,
                "size": size,
                "protein_option": "",  # Always default to empty string
                "protein_category": protein_category,
                "meal_types": meal_types,
                "query": query,
                "user_dislikes": user_dislikes
            }

    return user_preferences  # ✅ Return extracted user preferences


In [67]:
user_preferences

{'65e1d786f967c10fcba692fd': {'user_pref': 'Herb Mashed Potato, Cowboy Salad , Chicken a la King & Onion Bread Roll, Beetroot & Quinoa Salad, Coconut Curry and Quinoa, Blackened Protein with Braised Lentils',
  'user_likes': 'Mediterranean cuisine',
  'user_allergens': {'Lamb'},
  'size': 'Large',
  'protein_option': '',
  'protein_category': 'balance',
  'meal_types': {' dinner', 'lunch'},
  'query': 'Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Herb Mashed Potato, Cowboy Salad , Chicken a la King & Onion Bread Roll, Beetroot & Quinoa Salad, Coconut Curry and Quinoa, Blackened Protein with Braised Lentils',
  'user_dislikes': None},
 '6597f5f893f2206fb148f076': {'user_pref': 'Korean Style Wrap, Cajun Roasted Chicken Sandwich, Lime Chili Quesadillas , Satay & Jasmin Rice, Lemon Quinoa with Roasted Protein, Maftoul with Zucchini & Minced Protein . Highest Rated Dishes: Coconut Curry and Quinoa, Korean Style Wrap, Shawarma Bowl , Satay & Jasmin Rice, English Loaf',
  'use

### ***Recommendations**

In [74]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_allergens, user_dislikes, query, meal_category, size, protein_option, protein_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        # "size": {"$eq": size},  # Filter for specific size
        # "protein_option": {"$eq": protein_option},  # Filter for specific protein option
        # "protein_category": {"$eq": protein_category},  # Filter for specific protein category
        "allergens": {"$nin": list(user_allergens)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_allergens)}  # Exclude recipes containing allergens
    }
     # ✅ Add `size` filter only if it's not empty
    if size:
        pinecone_filter["size"] = {"$eq": size}

    # ✅ Add `protein_option` filter only if it's not empty
    if protein_category:
        pinecone_filter["protein_category"] = {"$eq": protein_category}

    # ✅ Add `protein_option` filter only if it's not empty
    if protein_option:
        pinecone_filter["protein_option"] = {"$eq": protein_option}
    # Retrieve documents from Pinecone with filtering for dislikes and allergens in ingredients
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k,  # Fetch only the required number of results
        filter=pinecone_filter  # Apply the filter for dislikes and allergens in ingredients
    )


    return docs


# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(filtered_docs, query, user_likes, user_pref, meal_types):
    prompt = f"""Generate a weekly meal plan based on the following user persona and recipe data:
    Only use the recipes exactly as they are presented below. Do not modify the recipes, add any extra information, or create new recipes. 
    Simply output the meal name for each day, using only the recipes provided. Do not alter or adapt the meals.
    May include those recipes that have disliked ingredients only if the recipes are not sufficient.
    If the available recipes are insufficient, you may include recipes that contain disliked ingredients. However, you must strictly follow these rules when checking for disliked ingredients:

    1. **ONLY check for disliked ingredients listed in the user's dislikes below. Do NOT flag any ingredient that is NOT in the dislikes list.**
    2. **You must NOT infer, assume, or guess the presence of any ingredient. Only check the explicit list of ingredients provided in the recipe.**
    3. **If a disliked ingredient (from the list below) is found in a meal, append the following notation to the meal: ' * (contains <disliked ingredient>)'.**
    4. **If a meal does not contain any disliked ingredients from the list below, do NOT append anything.**
    5. **Do NOT flag Butter, Cheddar Cheese, Yoghurt, Mascarpone Cheese, Mayonnaise, or any other ingredient unless they are explicitly listed in the dislikes section below and the ingredients list of the recipe.**
    6. **Ensure that the output is in JSON format only.**

    **Meals must be assigned to their respective categories: Breakfast for Breakfast, and Lunch and Dinner should be selected only from the Meal category, with evening_snacks and morning_snacks from Snacks.**
    **There are total 5 meal categories that include breakfast,morning_snack,lunch,evening_snack,dinner.Only include those meal categories that were mentioned by the user for each day of the week.Do not skip any meal type if it is present in the Meal Category**
    **Use only meals from the correct category:**
       - **Morning Snack:** Use recipes labeled `"snack"`.  
       - **Meals (Lunch/Dinner):** Use recipes labeled `"meal"`.  
       - **Evening Snack:** Use recipes labeled `"snack"`.
    **Do NOT repeat the same meal multiple times.**  
    Ensure the correct order of meals in the daily schedule:  
    1. **The output for each day must strictly follow this order (if selected):**  
    - **Morning Snack → Breakfast → Lunch → Dinner → Evening Snack**  
    2. **Do not skip any meal category that is in the user selection.**  

    **Use each recipe only once per week if possible.**  
    **Distribute meals evenly across all days to avoid repetition.**  
    **Rotate through the provided recipes to maximize diversity.**  
    **Do not repeat the same lunch or dinner within a week unless there are no other options.**  
    **If there are more recipes available than required, prioritize using each at least once before repeating any.**

    Give response in json format only.
        User Persona:
         Dietary Restrictions: {user_allergens}
         Dislikes: {user_dislikes}
         Likes: {user_likes} 
         Spice Level: Medium 
         Popular Dishes: {user_pref}
         Meal Frequency: {len(meal_types)}
         Meal Categories: {meal_types}
        """
    for i, doc in enumerate(filtered_docs, 1):
        metadata = doc.metadata

        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {metadata.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {metadata.get('description', 'No description')}\n"
        prompt += f" Ingredients: {', '.join(metadata.get('ingredients', []))}\n"
        prompt += f" Spice Level: {metadata.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {metadata.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {metadata.get('meal_category', 'Unknown')}\n"
        # prompt += f" Variants: {metadata.get('variants', 'Unknown')}\n"
    
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref, size, protein_option, protein_category, meal_types):
    # Define mapping of meal types to their respective counts
    meal_counts = {
        "breakfast": 8,
        "snack": 8,  # Each snack type (morning/evening) adds 8
        "meal": 0  # Lunch and Dinner are combined into "meal"
    }

    # Initialize recipe counts
    total_snack_count = 0
    total_meal_count = 0  # To hold combined lunch + dinner count
    fetched_recipes = {}

    # Determine the total number of snack recipes needed
    if "morning_snack" in meal_types or "evening_snack" in meal_types:
        total_snack_count = meal_counts["snack"] * sum(1 for meal in meal_types if "snack" in meal)

    # If lunch or dinner is included, sum their counts into "meal"
    if "lunch" in meal_types:
        total_meal_count += 10
    if "dinner" in meal_types:
        total_meal_count += 10

    # Fetch recipes for each meal type
    for meal_type in meal_types:
        if "snack" in meal_type or meal_type in ["lunch", "dinner"]:
            continue  # Skip individual snacks and meals; handle separately

        count = meal_counts.get(meal_type, 0)
        if count > 0:
            # Set size to "standard" for breakfast and snack
            meal_size = "standard" if meal_type in ["breakfast", "snack"] else size

            fetched_recipes[meal_type] = filter_recipes(
                vectorstore, user_allergens, user_dislikes, query, meal_type, meal_size, protein_option, protein_category, count
            )

    # Fetch total meal recipes (combined lunch & dinner) with user-specified size
    if total_meal_count > 0:
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, user_allergens, user_dislikes, query, "meal", size, protein_option, protein_category, total_meal_count
        )

    # Fetch total snack recipes (combined morning & evening snacks) with "standard" size
    if total_snack_count > 0:
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, user_allergens, user_dislikes, query, "snack", "standard", protein_option, protein_category, total_snack_count
        )

    # Debugging: Print fetched recipe counts
    for meal_type, recipes in fetched_recipes.items():
        print(f"{meal_type.capitalize()} recipes: {len(recipes)}")
    
    print("Fetching complete.")

    # Check if the total number of recipes is less than expected
    total_recipes = sum(len(recipes) for recipes in fetched_recipes.values())
    expected_total = sum(meal_counts.get(meal, 0) for meal in meal_types if meal != "snack") + total_snack_count + total_meal_count

    if total_recipes < expected_total:
        print("Not enough recipes found. Retrying without allergens filter.")
        user_allergens = {}  # Reset allergens and retry

        if total_meal_count > 0:
            fetched_recipes["meal"] = filter_recipes(
                vectorstore, user_allergens, user_dislikes, query, "meal", size, protein_option, protein_category, total_meal_count
            )

        if total_snack_count > 0:
            fetched_recipes["snack"] = filter_recipes(
                vectorstore, user_allergens, user_dislikes, query, "snack", "standard", protein_option, protein_category, total_snack_count
            )

    # Combine all selected recipes into the final list
    final_docs = [recipe for recipes in fetched_recipes.values() for recipe in recipes]
   
     # Step 3: Format the structured meal plan prompt
    final_prompt = format_meal_plan_prompt(final_docs, query, user_likes, user_pref, meal_types)

    # Step 4: Generate the response using LLM
    meal_plan = generate_response(final_prompt)

    print(meal_plan)
    return meal_plan, final_docs

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_allergens = {}  # Example allergens
    # user_dislikes = {
    #     "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    #     "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    #     "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    #     "White Fish", "Worcestershire Sauce"
    # }  # Example disliked ingredients
    
    # query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    # generate_meal_plan(vectorstore, user_allergens, user_dislikes, query, user_likes, user_pref)

In [81]:
# Call function to extract user preferences
user_preferences = extract_user_preferences()

# Iterate over all users and generate meal plans
for user_id, prefs in user_preferences.items():
    print(f"\nGenerating meal plan for User ID: {user_id}")

    # Extract user details
    user_allergens = prefs["user_allergens"] if prefs["user_allergens"] else {}  
    user_dislikes = prefs["user_dislikes"] if prefs["user_dislikes"] else {}  
    query = prefs["query"]
    user_likes = prefs["user_likes"]
    user_pref = prefs["user_pref"]
    size = prefs["size"].lower() if prefs["size"] else ""  # Convert to lowercase
    protein_option = prefs["protein_option"] if prefs["protein_option"] else ""  
    protein_category = prefs["protein_category"].lower() if prefs["protein_category"] else ""  # Convert to lowercase
    meal_types = prefs["meal_types"] if prefs["meal_types"] else set()  

    # Generate meal plan
    meal_plan, final_docs = generate_meal_plan(
        vectorstore,
        user_allergens,
        user_dislikes,
        query,
        user_likes,
        user_pref,
        size,
        protein_option,
        protein_category,
        meal_types
    )

    # Print the meal plan
    print(f"\nMeal Plan for User ID: {user_id}:\n{meal_plan}\n")



Processing: user_data/processed_files/processed_65e1d786f967c10fcba692fd.json

Processing: user_data/processed_files/processed_6597f5f893f2206fb148f076.json

Processing: user_data/processed_files/processed_679bd597ed9daf35d7885142.json

Processing: user_data/processed_files/processed_674ace1b814a540cda705dbe.json

Processing: user_data/processed_files/processed_66224ce5566a4aef142a7d41.json

Processing: user_data/processed_files/processed_64c2094757975b247fb1b63f.json

Processing: user_data/processed_files/processed_66acddb36777e93279d8b6db.json

Processing: user_data/processed_files/processed_65a677b8062f61836c3dbffc.json

Processing: user_data/processed_files/processed_64e77a0e5d497c8a38d8840a.json

Processing: user_data/processed_files/processed_66a799812279daa285f1d82e.json

Processing: user_data/processed_files/processed_64f48df85d497c8a3806286c.json

Processing: user_data/processed_files/processed_64c9f429cf697b0d786160ae.json

Processing: user_data/processed_files/processed_66a

ResourceExhausted: 429 Resource has been exhausted (e.g. check quota).

In [ ]:
import json
import csv
import os

# Sample JSON responses (Replace this with actual JSON extraction)
meal_plans = {
    "66acddb36777e93279d8b6db": {
        "weeklyMealPlan": {
            "Monday": {"lunch": "Greek Chicken with Herb Rice"},
            "Tuesday": {"lunch": "Tagliatelle Ragu"},
            "Wednesday": {"lunch": "English Loaf"},
            "Thursday": {"lunch": "Gyro Bowl"},
            "Friday": {"lunch": "Roasted Protein & Mash Potato"},
            "Saturday": {"lunch": "Lamb Stew with Colcannon Sweet Potato"},
            "Sunday": {}  # Empty day
        }
    },
    "65e1d786f967c10fcba692fd": {
        "weeklyMealPlan": {
            "Monday": {"lunch": "Beetroot & Quinoa Salad", "dinner": "Coconut Curry and Quinoa"},
            "Tuesday": {"lunch": "Cowboy Salad", "dinner": "Blackened Protein with Braised Lentils"},
            "Wednesday": {"lunch": "Herb Mashed Potato", "dinner": "Chicken a la King & Onion Bread Roll"},
            "Thursday": {"lunch": "Beetroot & Quinoa Salad", "dinner": "Cowboy Salad"},
            "Friday": {"lunch": "Blackened Protein with Braised Lentils", "dinner": "Coconut Curry and Quinoa"},
            "Saturday": {"lunch": "Chicken a la King & Onion Bread Roll", "dinner": "Herb Mashed Potato"},
            "Sunday": {"lunch": "Cowboy Salad", "dinner": "Blackened Protein with Braised Lentils"}
        }
    }
}

# Sample CSV customer data (Replace this with actual extracted data)
customer_data = {
    "66acddb36777e93279d8b6db": {
        "customer_id": "66acddb36777e93279d8b6db",
        "avoid_ingredient": "Cajun Spice, Chilli, Chilli Paste, Chipotle Chilli Paste, Gochujang Sauce, Green Chilli, Holland Chili, Hot Sauce, Red Chili, Red Chilli Powder, Thai Green Chilies",
        "diet_type": "balance",
        "plan": "lunch",
        "variant_size": "Medium",
        "customer_name": "Christine Maalouf"
    },
    "65e1d786f967c10fcba692fd": {
        "customer_id": "65e1d786f967c10fcba692fd",
        "avoid_ingredient": "Chicken, Chicken Stock Powder, Chicken Jus, Chicken Sausage",
        "diet_type": "vegetarian",
        "plan": "lunch, dinner",
        "variant_size": "Large",
        "customer_name": "John Doe"
    }
}

# Define CSV headers
headers = ["customer_id", "avoid_ingredient", "diet_type", "plan", "variant_size", "customer_name",
           "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

# Convert JSON meal plan to CSV format
csv_data = []

for user_id, meal_plan in meal_plans.items():
    if user_id in customer_data:
        user_info = customer_data[user_id]
        row = {
            "customer_id": user_info["customer_id"],
            "avoid_ingredient": user_info["avoid_ingredient"],
            "diet_type": user_info["diet_type"],
            "plan": user_info["plan"],
            "variant_size": user_info["variant_size"],
            "customer_name": user_info["customer_name"]
        }

        # Fill meals for each day
        for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
            meals = meal_plan["weeklyMealPlan"].get(day, {})
            formatted_meals = ", ".join([f"{meal_type} - {meal}" for meal_type, meal in meals.items()])
            row[day.lower()] = formatted_meals if formatted_meals else ""

        csv_data.append(row)

# Write to CSV file
csv_filename = "converted_meal_plans.csv"
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=headers)
    writer.writeheader()
    writer.writerows(csv_data)

print(f"Meal plans successfully saved to {csv_filename}")


In [ ]:

"""


Meal Plan for User ID: 6597f5f893f2206fb148f076:
```json
{
  "monday": {
    "lunch": "Korean Style Wrap",
    "dinner": "Coconut Curry and Quinoa"
  },
  "tuesday": {
    "lunch": "Satay & Jasmin Rice",
    "dinner": "Asian Meatballs with Fried Rice"
  },
  "wednesday": {
    "lunch": "Southwest Tacos",
    "dinner": "Coconut Curry and Quinoa"
  },
  "thursday": {
    "lunch": "Souvlaki & Quinoa Pilaf",
    "dinner": "Greek Chicken with Herb Rice"
  },
  "friday": {
    "lunch": "English Loaf",
    "dinner": "Southwest Tacos"
  }
}
```






Meal Plan for User ID: 679bd597ed9daf35d7885142:
```json
{
  "monday": {
    "lunch": "Orzo Salad & Maple Lemon Dressing",
    "dinner": "Greek Chicken with Herb Rice"
  },
  "tuesday": {
    "lunch": "Asian Meatballs with Fried Rice",
    "dinner": "Korean Style Wrap"
  },
  "wednesday": {
    "lunch": "Satay & Jasmin Rice",
    "dinner": "Gyro Bowl * (contains Jalapeno)"
  },
  "thursday": {
    "lunch": "Asian Meatballs with Fried Rice",
    "dinner": "Souvlaki & Quinoa Pilaf * (contains Quinoa)"
  },
  "friday": {
    "lunch": "Gyro Bowl * (contains Jalapeno)",
    "dinner": "Cowboy Salad"
  }
}
```






Meal Plan for User ID: 674ace1b814a540cda705dbe:
```json
{
  "monday": {
    "morning_snack": "Omega Egg Protein Pot",
    "lunch": "Chicken Crunchy Bowl",
    "dinner": "Cheese and Nuts Pot"
  },
  "tuesday": {
    "morning_snack": "Crudites & Sour Cream Dip",
    "lunch": "Roasted Pecans",
    "dinner": "Buffalo Chicken Salad * (contains Bell Pepper)"
  },
  "wednesday": {
    "morning_snack": "Cheese and Nuts Pot",
    "lunch": "Omega Egg Protein Pot",
    "dinner": "Chicken Crunchy Bowl"
  },
  "thursday": {
    "morning_snack": "Buffalo Chicken Salad * (contains Bell Pepper)",
    "lunch": "Roasted Pecans",
    "dinner": "Crudites & Sour Cream Dip"
  },
  "friday": {
    "morning_snack": "Roasted Pecans",
    "lunch": "Crudites & Sour Cream Dip",
    "dinner": "Buffalo Chicken Salad * (contains Bell Pepper)"
  },
  "saturday": {},
  "sunday": {}
}
```






Meal Plan for User ID: 66224ce5566a4aef142a7d41:
```json
{
  "monday": {
    "dinner": "Souvlaki & Quinoa Pilaf"
  },
  "tuesday": {
    "dinner": "Greek Chicken with Herb Rice"
  },
  "wednesday": {
    "dinner": "Asian Meatballs with Fried Rice"
  },
  "thursday": {
    "dinner": "Roasted Protein & Mash Potato"
  },
  "friday": {
    "dinner": "English Loaf"
  },
  "saturday": {
    "dinner": "Creamy Quinoa Bowl"
  },
  "sunday": {
    "dinner": "Gyro Bowl"
  }
}
```






Meal Plan for User ID: 64c2094757975b247fb1b63f:
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "lunch": "Souvlaki & Quinoa Pilaf",
      "dinner": "Greek Chicken with Herb Rice"
    },
    "Tuesday": {
      "lunch": "Gyro Bowl",
      "dinner": "Creamy Quinoa Bowl"
    },
    "Wednesday": {
      "lunch": "Souvlaki & Quinoa Pilaf",
      "dinner": "Coconut Curry and Quinoa"
    },
    "Thursday": {
      "lunch": "Gyro Bowl",
      "dinner": "Asian Meatballs with Fried Rice"
    },
    "Friday": {
      "lunch": "Creamy Quinoa Bowl",
      "dinner": "Coconut Curry and Quinoa"
    },
    "Saturday": {
      "lunch": "Greek Chicken with Herb Rice",
      "dinner": "Asian Meatballs with Fried Rice"

    },
    "Sunday": {
      "lunch": "Souvlaki & Quinoa Pilaf",
      "dinner": "Creamy Quinoa Bowl"
    }
  }
}
```






Meal Plan for User ID: 66acddb36777e93279d8b6db:
```json
{
  "Monday": {
    "lunch": "Souvlaki & Quinoa Pilaf"
  },
  "Tuesday": {
    "lunch": "Greek Chicken with Herb Rice"
  },
  "Wednesday": {
    "lunch": "Gyro Bowl"
  },
  "Thursday": {
    "lunch": "Orzo Salad & Maple Lemon Dressing"
  },
  "Friday": {
    "lunch": "Souvlaki & Quinoa Pilaf"
  },
  "Saturday": {
    "lunch": "English Loaf"
  },
  "Sunday": {
    "lunch": "Asian Meatballs with Fried Rice * (contains Gochujang Paste)"
  }
}
```






Meal Plan for User ID: 65a677b8062f61836c3dbffc:
```json
{
  "monday": {
    "lunch": "Shawarma Bowl",
    "dinner": "Greek Chicken with Herb Rice"
  },
  "tuesday": {
    "lunch": "Orzo Salad & Maple Lemon Dressing",
    "dinner": "Gyro Bowl"
  },
  "wednesday": {
    "lunch": "Souvlaki & Quinoa Pilaf",
    "dinner": "Shawarma Bowl"
  },
  "thursday": {
    "lunch": "Satay & Jasmin Rice",
    "dinner": "Asian Meatballs with Fried Rice"
  },
  "friday": {
    "lunch": "Gyro Bowl * (contains Yoghurt)",
    "dinner": "Satay & Jasmin Rice"
  }
}
```






Meal Plan for User ID: 64e77a0e5d497c8a38d8840a:
```json
{
  "monday": {
    "dinner": "Greek Chicken with Herb Rice"
  },
  "tuesday": {
    "dinner": "Asian Meatballs with Fried Rice"
  },
  "wednesday": {
    "dinner": "Satay & Jasmin Rice"
  },
  "thursday": {
    "dinner": "English Loaf"
  },
  "friday": {
    "dinner": "Asian Meatballs with Fried Rice"
  },
  "saturday": {
    "dinner": "Greek Chicken with Herb Rice"
  },
  "sunday": {
    "dinner": "Satay & Jasmin Rice"
  }
}
```






Meal Plan for User ID: 66a799812279daa285f1d82e:
```json
{
  "monday": {
    "dinner": "Shawarma Bowl"
  },
  "tuesday": {
    "dinner": "Orzo Salad & Maple Lemon Dressing"
  },
  "wednesday": {
    "dinner": "Gyro Bowl"
  },
  "thursday": {
    "dinner": "Greek Chicken with Herb Rice"
  },
  "friday": {
    "dinner": "Souvlaki & Quinoa Pilaf"
  },
  "saturday": {
    "dinner": "Satay & Jasmin Rice"
  },
  "sunday": {
    "dinner": "Cowboy Salad"
  }
}
```






Meal Plan for User ID: 64f48df85d497c8a3806286c:
```json
{
  "monday": {
    "lunch": "Greek Chicken with Herb Rice"
  },
  "tuesday": {
    "lunch": "Gyro Bowl"
  },
  "wednesday": {
    "lunch": "Souvlaki & Quinoa Pilaf"
  },
  "thursday": {
    "lunch": "Orzo Salad & Maple Lemon Dressing"
  },
  "friday": {
    "lunch": "Gyro Bowl"
  },
  "saturday": {
    "lunch": "Souvlaki & Quinoa Pilaf"
  },
  "sunday": {
    "lunch": "Creamy Quinoa Bowl"
  }
}
```






Meal Plan for User ID: 64c9f429cf697b0d786160ae:
```json
{
  "monday": {
    "lunch": "Creamy Quinoa Bowl",
    "dinner": "Asian Meatballs with Fried Rice"
  },
  "tuesday": {
    "lunch": "Greek Chicken with Herb Rice",
    "dinner": "English Loaf"
  },
  "wednesday": {
    "lunch": "Southwest Tacos",
    "dinner": "Creamy Quinoa Bowl"
  },
  "thursday": {
    "lunch": "Gyro Bowl",
    "dinner": "Southwest Tacos"
  },
  "friday": {
    "lunch": "English Loaf",
    "dinner": "Southwest Tacos"
  }
}
```






Meal Plan for User ID: 66a934422279daa2850029ae:
```json
{
  "monday": {
    "lunch": "Souvlaki & Quinoa Pilaf",
    "dinner": "Shawarma Bowl"
  },
  "tuesday": {
    "lunch": "Greek Chicken with Herb Rice",
    "dinner": "Southwest Tacos"
  },
  "wednesday": {
    "lunch": "Southwest Tacos",
    "dinner": "Souvlaki & Quinoa Pilaf"
  },
  "thursday": {
    "lunch": "Greek Chicken with Herb Rice",
    "dinner": "English Loaf"
  },
  "friday": {
    "lunch": "Southwest Tacos",
    "dinner": "Shawarma Bowl"
  }
}
```


"""

'Meal Plan for User ID: 65e1d786f967c10fcba692fd:\n```json\n{\n  "monday": {\n    "lunch": "Souvlaki & Quinoa Pilaf",\n    "dinner": "Coconut Curry and Quinoa"\n  },\n  "tuesday": {\n    "lunch": "Blackened Protein with Braised Lentils",\n    "dinner": "Cowboy Salad"\n  },\n  "wednesday": {\n    "lunch": "Souvlaki & Quinoa Pilaf",\n    "dinner": "Blackened Protein with Braised Lentils"\n  },\n  "thursday": {\n    "lunch": "Coconut Curry and Quinoa",\n    "dinner": "Cowboy Salad" \n  },\n  "friday": {\n    "lunch": "Blackened Protein with Braised Lentils",\n    "dinner": "English Loaf"\n  },\n  "saturday": {\n    "lunch": "Cowboy Salad",\n    "dinner": "Coconut Curry and Quinoa"\n  },\n  "sunday": {\n    "lunch": "Blackened Protein with Braised Lentils",\n    "dinner": "Souvlaki & Quinoa Pilaf"\n  }\n}\n```\nGenerating meal plan for User ID: 6597f5f893f2206fb148f076\nMeal recipes: 10\nFetching complete.\n```json\n{\n  "monday": {\n    "lunch": "Korean Style Wrap",\n    "dinner": "Coconu